# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step walk-through for loading, exploring, and processing a FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}:\n{metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, their `@id` values, and key information about them.

We will enumerate record sets and, for each, their available fields/columns, always referencing each by its `@id`.

In [ ]:
# List all record sets defined in the dataset
record_sets = list(metadata.record_sets)
print(f"Record sets in dataset ({len(record_sets)} total):\n")

for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    # Fields in the record set
    field_ids = [f.id for f in rs.fields]
    print(f"  Fields (@id):")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) [dataType: {field.data_type}]")
    # Show column IDs if explicitly defined via columns/column
    if hasattr(rs, 'columns') and rs.columns:
        print(f"  Columns:")
        for col in rs.columns:
            print(f"    - {col.id}")
    print()

Next, let's see a sample of records from each record set, always using `@id` reference.

In [ ]:
# Show first few records for each record set using `@id`
for rs in record_sets:
    print(f"Sample records from RecordSet '{rs.name}' (@id: {rs.id}):")
    try:
        for idx, record in enumerate(dataset.records(record_set=rs.id)):
            print(f" {record}")
            if idx >= 2:  # Only show first 3 records
                break
    except Exception as e:
        print(f"  Could not load records or record set is empty. Reason: {e}")
    print()

## 3. Data Extraction
Load data from each record set into DataFrames for analysis. We'll use the record set and field `@id`s from above.

For demonstration, we extract each non-empty record set as a separate dataframe, referenced by its `@id`.

In [ ]:
dataframes = {}
loaded_record_set_ids = []

# Extract all non-empty record sets
for rs in record_sets:
    try:
        records = list(dataset.records(record_set=rs.id))
        if records:  # Only add if non-empty
            dataframes[rs.id] = pd.DataFrame(records)
            loaded_record_set_ids.append(rs.id)
            print(f"Loaded {len(records)} records from RecordSet '{rs.name}' (@id: {rs.id})")
    except Exception as e:
        print(f"Skipping RecordSet '{rs.name}' (@id: {rs.id}): {e}")

if dataframes:
    first_rs_id = loaded_record_set_ids[0]
    print(f"\nColumns in first loaded RecordSet (@id: {first_rs_id}):\n{dataframes[first_rs_id].columns.tolist()}")
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
To illustrate typical EDA, we'll:

- Select a numeric field using its `@id` (from the previous overview; choose a field matching `Float`, `Integer`, etc.)
- Filter records where the field value passes a certain threshold
- Normalize that column
- Optionally group by a categorical field, referenced by its `@id`.

In [ ]:
# Adjust these to match actual @id values from your dataset

# If no record sets loaded, stop
if not dataframes:
    print("No dataframes loaded to analyze.")
else:
    # Pick first available record set and numeric field
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Attempt to select a numeric field by inspecting dtype or searching for typical names
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    
    if numeric_field_id is None:
        print(f"No numeric field found in RecordSet (@id: {record_set_id}).")
    else:
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (using `@id`):")
        display(filtered_df.head())

        # Normalize field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to find a grouping/categorical field (non-numeric)
        group_field_id = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() < 10:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped data by {group_field_id} (using @id):")
            display(grouped_df.head())
        else:
            print("\nNo suitable grouping field (categorical, low cardinality) found.")

## 5. Visualization
Visualize data distributions or relationships between fields.

Below, we plot the distribution of the selected numeric field and compare it across group values, if a grouping field was found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or numeric_field_id is None:
    print("No data or numeric field to plot.")
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Compare numeric field by group if possible
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:

- Load and introspect a FAIR^2 dataset defined by a Croissant schema using `mlcroissant`
- Enumerate record set, field, and column `@id`s for rigorous and consistent referencing
- Extract and manipulate records for exploratory analysis
- Visualize field distributions and relationships for further scientific investigation

All references to data entities were made exclusively via their `@id` values, which ensures reproducible and transparent dataset operations in accordance with FAIR principles. For further analysis, consult the schema and data documentation for the full range of available attributes.